# Qwen Portable Slice Row-Processing Proof (Colab GPU)

This notebook runs the localized portable-slice Colab proof for the Task 103 row-processing lane.

Invariants:
- Hemma remains the only place that selects rows.
- This notebook consumes one Hemma-issued portable slice bundle.
- The notebook is only an orchestrator around repo-owned script surfaces.
- Output must match the canonical Task 103 row-processing run-root contract.


## Hemma preparation (run before opening this notebook)

Use a fresh proof-only `source-selection` universe, not the live Hemma `10k` run:

```bash
pdm run run-hemma -- pdm run task-103-preprocess-public-corpus launch \
  --task103-stage source-selection \
  --launch-id task121-colab-proof-selection-launch-20260310a \
  --task103-run-id task121-colab-proof-selection-20260310a \
  --rixvox-split train \
  --rixvox-max-rows-per-split 512 \
  --skip-build

pdm run run-hemma -- pdm run task-121-colab-slice-bundle plan \
  --source-run-root /srv/scratch/sir-convert-a-lot/build/runs/qwen3-tts-swedish-preprocessing/task121-colab-proof-selection-20260310a \
  --output-root /srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task121-proof-slice-1-of-2-20260310a \
  --slice-count 2 \
  --slice-index 1
```

Then make these three files available under `SLICE_ROOT` in Colab:
- `selected_source_records.jsonl`
- `required_hub_files.json`
- `slice_summary.json`


In [ ]:
%pip install -q accelerate datasets huggingface_hub jiwer librosa \
    onnxruntime pyarrow python-dotenv qwen-tts safetensors \
    sentencepiece soundfile sox transformers

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchaudio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio"],
        check=True,
    )


In [ ]:
import json
import os
import subprocess
import sys
import tarfile
import time
from pathlib import Path


def _candidate_repo_roots() -> list[Path]:
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)
    candidates.append(Path("/content/sir-convert-a-lot"))
    return candidates


def _resolve_repo_root() -> Path:
    for candidate in _candidate_repo_roots():
        if (candidate / "scripts" / "sir_convert_a_lot").exists():
            return candidate
    target = Path("/content/sir-convert-a-lot")
    if not target.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/olofsg/sir-convert-a-lot.git",
                target.as_posix(),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "fetch", "origin", "main"], check=True, cwd=target)
        subprocess.run(["git", "checkout", "main"], check=True, cwd=target)
        subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True, cwd=target)
    return target


REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
SLICE_ROOT = REPO_ROOT / "colab_inputs" / "task121-proof-slice-1-of-2-20260310a"
PROOF_BUNDLE_PATH = (
    REPO_ROOT
    / "colab_ml_training"
    / "proof_inputs"
    / "task122-proof-slice-1-of-2-20260310a-bundle.tar.gz"
)
DATA_ROOT = Path("/content/data/qwen3-tts-swedish-corpus")
RUN_ROOT = Path("/content/work/runs/task121-colab-proof-rowproc-20260310a")
OUTPUT_ROOT = Path("/content/work/reference/qwen3-tts-swedish-corpus")
CACHE_DIR = Path("/content/cache/huggingface")
ROWPROC_STDOUT_PATH = RUN_ROOT / "row_processing.stdout.log"
ROWPROC_STDERR_PATH = RUN_ROOT / "row_processing.stderr.log"
LOCALIZED_SELECTED_SOURCE_RECORDS_PATH = SLICE_ROOT / "localized_selected_source_records.jsonl"
ROWPROC_TIMEOUT_SECONDS = 3 * 60 * 60
ROW_WORKER_COUNT = 8
GPU_ASR_WORKER_COUNT = 2

for path in (SLICE_ROOT, DATA_ROOT, RUN_ROOT.parent, OUTPUT_ROOT.parent, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "repo_root": REPO_ROOT.as_posix(),
    "slice_root": SLICE_ROOT.as_posix(),
    "proof_bundle_path": PROOF_BUNDLE_PATH.as_posix(),
    "data_root": DATA_ROOT.as_posix(),
    "run_root": RUN_ROOT.as_posix(),
    "output_root": OUTPUT_ROOT.as_posix(),
    "cache_dir": CACHE_DIR.as_posix(),
    "rowproc_stdout_path": ROWPROC_STDOUT_PATH.as_posix(),
    "rowproc_stderr_path": ROWPROC_STDERR_PATH.as_posix(),
    "localized_selected_source_records_path": LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.as_posix(),
    "rowproc_timeout_seconds": ROWPROC_TIMEOUT_SECONDS,
    "row_worker_count": ROW_WORKER_COUNT,
    "gpu_asr_worker_count": GPU_ASR_WORKER_COUNT,
}, indent=2))


In [ ]:
if not PROOF_BUNDLE_PATH.exists():
    raise FileNotFoundError(
        "Expected committed proof bundle at " + PROOF_BUNDLE_PATH.as_posix()
    )

with tarfile.open(PROOF_BUNDLE_PATH, "r:gz") as archive:
    archive.extractall(SLICE_ROOT)

required_bundle_files = [
    SLICE_ROOT / "selected_source_records.jsonl",
    SLICE_ROOT / "required_hub_files.json",
    SLICE_ROOT / "slice_summary.json",
]
missing_bundle_files = [
    path.as_posix() for path in required_bundle_files if not path.exists()
]
if missing_bundle_files:
    raise FileNotFoundError(
        "Portable slice bundle extraction failed: "
        + ", ".join(missing_bundle_files)
    )

slice_summary = json.loads((SLICE_ROOT / "slice_summary.json").read_text(encoding="utf-8"))
slice_summary


## Stage required raw files and localize the slice

This uses only repo-owned portable-slice surfaces: first stage the exact Hub files listed in `required_hub_files.json`, then localize the selected slice into plain local audio files plus a persisted localized selected-source manifest.


In [ ]:
stage_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle",
    "stage-required-files",
    "--slice-root",
    str(SLICE_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--cache-dir",
    str(CACHE_DIR),
]
localize_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle",
    "localize-slice",
    "--slice-root",
    str(SLICE_ROOT),
    "--data-root",
    str(DATA_ROOT),
]
print(" ".join(stage_command))
subprocess.run(stage_command, check=True, cwd=REPO_ROOT)
print(" ".join(localize_command))
subprocess.run(localize_command, check=True, cwd=REPO_ROOT)
assert LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.exists(), (
    "Expected localized selected-source manifest at "
    + LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.as_posix()
)


## Run canonical Task 103 row-processing on the portable slice

This is the actual proof step. It must emit the same Task 103 run-root structure as Hemma row-processing while using the Colab GPU with one bounded but nontrivial ASR concurrency lane.


In [ ]:
def _read_status_payload() -> dict[str, object] | None:
    status_path = RUN_ROOT / "status.json"
    if not status_path.exists():
        return None
    return json.loads(status_path.read_text(encoding="utf-8"))


def _spool_count() -> int:
    return sum(1 for _ in RUN_ROOT.rglob("spool/rows/**/*.json"))


def _audio_count() -> int:
    return sum(1 for _ in RUN_ROOT.rglob("audio_24k/**/*.wav"))


def _tail(path: Path, line_count: int = 80) -> str:
    if not path.exists():
        return "<missing>"
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-line_count:])


rowproc_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.run_task103_qwen_swedish_preprocessing",
    "--source-mode",
    "selected-source-records",
    "--selected-source-records-path",
    str(LOCALIZED_SELECTED_SOURCE_RECORDS_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--run-root",
    str(RUN_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--stage",
    "row-processing",
    "--row-worker-count",
    str(ROW_WORKER_COUNT),
    "--gpu-asr-worker-count",
    str(GPU_ASR_WORKER_COUNT),
    "--resume-row-processing",
]
print(" ".join(rowproc_command))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
with (
    ROWPROC_STDOUT_PATH.open("w", encoding="utf-8") as stdout_handle,
    ROWPROC_STDERR_PATH.open("w", encoding="utf-8") as stderr_handle,
):
    process = subprocess.Popen(
        rowproc_command,
        cwd=REPO_ROOT,
        stdout=stdout_handle,
        stderr=stderr_handle,
        text=True,
    )
    started_at = time.time()
    while True:
        returncode = process.poll()
        status_payload = _read_status_payload()
        print(
            json.dumps(
                {
                    "elapsed_seconds": round(time.time() - started_at, 1),
                    "returncode": returncode,
                    "status": (
                        None if status_payload is None else status_payload.get("status")
                    ),
                    "processed_row_count": (
                        None
                        if status_payload is None
                        else status_payload.get("processed_row_count")
                    ),
                    "total_row_count": (
                        None
                        if status_payload is None
                        else status_payload.get("total_row_count")
                    ),
                    "current_dataset_row_id": (
                        None
                        if status_payload is None
                        else status_payload.get("current_dataset_row_id")
                    ),
                    "spool_rows": _spool_count(),
                    "audio_24k_files": _audio_count(),
                },
                indent=2,
            )
        )
        if returncode is not None:
            if returncode != 0:
                stdout_tail = _tail(ROWPROC_STDOUT_PATH)
                stderr_tail = _tail(ROWPROC_STDERR_PATH)
                raise RuntimeError(
                    "Row-processing failed.\nSTDOUT tail:\n"
                    + stdout_tail
                    + "\n\nSTDERR tail:\n"
                    + stderr_tail
                )
            break
        if time.time() - started_at > ROWPROC_TIMEOUT_SECONDS:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            stdout_tail = _tail(ROWPROC_STDOUT_PATH)
            stderr_tail = _tail(ROWPROC_STDERR_PATH)
            raise TimeoutError(
                "Row-processing timed out.\nSTDOUT tail:\n"
                + stdout_tail
                + "\n\nSTDERR tail:\n"
                + stderr_tail
            )
        time.sleep(20)
print({"elapsed_seconds": round(time.time() - started_at, 2)})


In [ ]:
status_payload = json.loads((RUN_ROOT / "status.json").read_text(encoding="utf-8"))
run_payload = json.loads((RUN_ROOT / "run.json").read_text(encoding="utf-8"))
spool_count = sum(1 for _ in RUN_ROOT.rglob("spool/rows/**/*.json"))
audio_count = sum(1 for _ in RUN_ROOT.rglob("audio_24k/**/*.wav"))
print(json.dumps({
    "status": status_payload,
    "run": run_payload,
    "spool_rows": spool_count,
    "audio_24k_files": audio_count,
}, indent=2))


## Success criteria

A successful first Colab proof should show:
- valid `run.json`
- valid `status.json`
- non-empty `inventory/`
- non-empty `audio_24k/`
- non-empty `spool/rows/`
- no notebook-only preprocessing logic outside these repo-owned commands
